# Bulk and Conservation Diagnostics for Temperature Forcing Runs

In [ ]:
import warnings
from functools import partial

import iris
import matplotlib.colors as mcol
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import numpy as np
import paths
import seaborn as sns
from aeolus.calc import (
    air_density,
    integrate,
    last_n_day_mean,
    spatial,
    toa_net_energy,
    zonal_mean,
)
from aeolus.const import init_const
from aeolus.coord import get_cube_rel_days, interp_cube_from_height_to_pressure_levels
from aeolus.lfric import fix_time_coord, load_lfric_raw, ugrid_spatial
from aeolus.model import lfric, um
from aeolus.plot import (
    add_custom_legend,
    figsave,
    map_scatter,
    stream,
    subplot_label_generator,
    tex2cf_units,
    timeseries_1d,
    timeseries_2d,
)
from common import (
    CATEGORIES,
    EXPERIMENTS,
    GROUPS,
    LATLON_GLOB,
    Diag,
    Experiment,
    lfric_callback_uniform_height,
)
from tqdm import tqdm

warnings.filterwarnings("ignore")
iris.FUTURE.date_microseconds = True

print(*EXPERIMENTS.keys(), sep="\n")

In [ ]:
group_key = "dhj_all"

runs = {}
for run_key in tqdm(GROUPS[group_key].simulations):
    runs[run_key] = load_lfric_raw(
        sorted(
            (
                paths.data_work
                / "lfric"
                / EXPERIMENTS[run_key].group
                / EXPERIMENTS[run_key].label
            ).glob("lfric_diag_cnsrv*.nc")
        )
        + sorted(
            (
                paths.data_work
                / "lfric"
                / EXPERIMENTS[run_key].group
                / EXPERIMENTS[run_key].label
            ).glob(LATLON_GLOB)
        ),
        callback=partial(
            lfric_callback_uniform_height,
            model_top_height=EXPERIMENTS[run_key].const.domain_height.data,
        ),
    )
    runs[run_key].append(
        load_lfric_raw(
            paths.data_work
            / "lfric"
            / EXPERIMENTS[run_key].group
            / EXPERIMENTS[run_key].label
            / "lfric_initial*.nc"
        ).extract_cube("grid_cell_area")
    )
print(run_key)
print(runs[run_key])
# ds_main = load_lfric_raw(
#     sorted(
#         (
#             paths.data_work
#             / "lfric"
#             / EXPERIMENTS[run_key].group
#             / EXPERIMENTS[run_key].label
#         ).glob("lfric_diag_main*.nc")
#     )[:],
#     callback=partial(
#         lfric_callback_uniform_height,
#         model_top_height=EXPERIMENTS[run_key].const.domain_height.data,
#     ),
# )
# print(ds_main)

# dens = air_density(ds_main, const=EXPERIMENTS[run_key].const).interpolate(
#     [(lfric.z, aam.coord(lfric.z).points)], iris.analysis.Linear()
# )
# dens_tot = ugrid_spatial(dens, "sum")
# aam_tot_vint_mw = (
#     integrate(aam_tot * dens_tot, "level_height") / integrate(dens_tot, "level_height")
# )
# print(aam_tot_vint_mw.units)

In [ ]:
DIAGS = {
    # "mass": Diag(
    #     title="Total Atmosphere Mass",
    #     units="$kg$ $m^{-2}$",
    #     recipe=lambda cl, const: ugrid_spatial(
    #         cl.extract_cube("atmosphere_mass_of_air_per_unit_area"), "sum"
    #     ),
    # ),
    "u_zm_max": Diag(
        title="Peak Zonal-Mean Wind",
        units="$km$ $s^{-1}$",
        recipe=lambda cl, const: spatial(
            zonal_mean(cl.extract_cube("eastward_wind")), "max"
        ).collapsed("level_height", iris.analysis.MAX),
    ),
    "ke": Diag(
        title="Total Kinetic Energy",
        units="$10^{27}$ $J$",
        recipe=lambda cl, const: ugrid_spatial(
            iris.analysis.maths.multiply(
                cl.extract_cube("atmosphere_vertical_kinetic_energy_content")
                + cl.extract_cube("atmosphere_horizontal_kinetic_energy_content"),
                iris.coords.AuxCoord(
                    cl.extract_cube("grid_cell_area").data, units="m2"
                ),
                dim=1,
            ),
            "sum",
        ),
    ),
    "aam_vint": Diag(
        title="Total Axial Angular Momentum",
        units="$kg$ $s^{-1}$",
        recipe=lambda cl, const: (
            integrate(
                ugrid_spatial(
                    iris.analysis.maths.multiply(
                        cl.extract_cube("axial_angular_momentum_at_cell_centres"),
                        iris.coords.AuxCoord(1.0, units="m2"),
                        dim=2,
                    ),
                    "sum",
                ),
                "level_height",
            )
            / const.domain_height
        ),
    ),
}

In [ ]:
normalise = {"aam_vint": "init"}
# normalise = {"aam_vint": "max"}
_savefig = True

fig = plt.figure(figsize=(10, 2), tight_layout=True)
# fig.suptitle("Conservation diagnostics", y=1.15)
axd = fig.subplot_mosaic(
    [DIAGS.keys()],
    # gridspec_kw={"wspace": 0.5},
    sharex=True,
)

iletters = subplot_label_generator()
# for ax in axd.values():
#     ax.set_title(f"", loc="left")

for diag_key, diag_prop in DIAGS.items():
    ax = axd[diag_key]
    units = tex2cf_units(diag_prop.units)
    ttl = f"({next(iletters)}) {diag_prop.title}"
    if diag_key in normalise:
        if normalise[diag_key] == "init":
            ttl_note = "normalised by\nthe initial value"
        elif normalise[diag_key] == "max":
            ttl_note = "normalised by\nthe maximum value"
        ax.add_artist(
            AnchoredText(
                ttl_note,
                loc="upper left" if group_key == "shj" else "lower left",
                prop={"size": "small"},
                frameon=False,
            )
        )
    else:
        if units.startswith("10"):
            ttl += rf" [$\times${diag_prop.units}]"
        else:
            ttl += f" [{diag_prop.units}]"
    ax.set_title(ttl, size="small")
    ax.set_xlim(0, 1200)
    ax.set_xticks(np.arange(0, 1201, 200))
    if ax.get_subplotspec().is_last_row():
        ax.set_xlabel("Simulation time [day]")
    # ax.spines["right"].set_visible(False)
    # ax.spines["top"].set_visible(False)
    # ax.grid(color="w")
    # ax.set_facecolor("#EEEEEE")
    sns.despine(ax=ax, offset=10)

    for run_key in GROUPS[group_key].simulations:
        try:
            cube = diag_prop.recipe(runs[run_key], const=EXPERIMENTS[run_key].const)
        except KeyError as e:
            print(e)
            continue
        if not (diag_key in normalise):
            cube.convert_units(units)

        x = get_cube_rel_days(cube)
        y = cube.copy().data
        if diag_key in normalise:
            if normalise[diag_key] == "init":
                y /= y[0]
            elif normalise[diag_key] == "max":
                y /= y.max()
            # y /= np.linalg.norm(y, ord=1)
            # y /= (sum(y**2))**0.5
        ax.plot(x, y, **EXPERIMENTS[run_key].kw_plt)

add_custom_legend(
    fig,
    {
        EXPERIMENTS[exp_key].title: {**EXPERIMENTS[exp_key].kw_plt}
        for exp_key in GROUPS[group_key].simulations
    },
    title=GROUPS[group_key].title,
    ncol=len(GROUPS[group_key].simulations),
    loc="upper center",
    bbox_to_anchor=(0.5, 1.2),
    # facecolor="#EEEEEE",
    edgecolor="none",
)

if _savefig:
    figsave(
        fig,
        paths.drafts / (f"cnsrv_diags__{group_key}__{'_'.join(DIAGS.keys())}"),
    )